# First-look at IMAP/HIT data.

A shorscript for getting oriented with the HIT l2 and l3 files. 

- Data was downloaded from https://imap.princeton.edu/data
- Under "Index of /imap/" select "hit/"




In [ ]:
# 1. Setup

from pathlib import Path
import cdflib
import pandas as pd

data_dir = Path(r"C:\Users\pauls\SEPs\IMAP\data\IMAP_30072026")   # edit path

pd.set_option("display.max_rows", None, "display.max_colwidth", 80)

for f in sorted(data_dir.glob("*.cdf")):
    print(f.name)

In [ ]:
# 2. Choose files to eamine

l2_path = sorted(data_dir.glob("*_l2_*.cdf"))[0] # change "0" to pick a different file
l3_path = sorted(data_dir.glob("*_l3_*.cdf"))[0] # change "0" to pick a different file
print("L2:", l2_path.name)
print("L3:", l3_path.name)

cdf_l2 = cdflib.CDF(str(l2_path))
cdf_l3 = cdflib.CDF(str(l3_path))

In [ ]:
# 3. Helper functions

def _field(obj, name):
    """cdflib returns dicts in older versions and objects in newer ones."""
    return obj[name] if isinstance(obj, dict) else getattr(obj, name)

def variable_names(cdf):
    info = cdf.cdf_info()
    return list(_field(info, "zVariables")) + list(_field(info, "rVariables"))

def variable_table(cdf):
    """One row per variable, from metadata only (no data loaded)."""
    rows = []
    for v in variable_names(cdf):
        atts = cdf.varattsget(v)
        inq = cdf.varinq(v)
        dims = tuple(_field(inq, "Dim_Sizes"))
        n_rec = _field(inq, "Last_Rec") + 1
        shape = (n_rec,) + dims if _field(inq, "Rec_Vary") else dims
        rows.append({
            "variable": v,
            "shape": shape,
            "units": atts.get("UNITS", ""),
            "description": atts.get("CATDESC", atts.get("FIELDNAM", "")),
            "depend_0": atts.get("DEPEND_0", ""),
            "depend_1": atts.get("DEPEND_1", ""),
        })
    return pd.DataFrame(rows)

def energy_bins(cdf, species):
    """Bin edges for one species. Handles L2 ('*_energy_mean') and L3 ('*_energy')."""
    names = set(variable_names(cdf))
    centre = f"{species}_energy_mean" if f"{species}_energy_mean" in names else f"{species}_energy"
    mean = cdf.varget(centre)
    dm = cdf.varget(f"{species}_energy_delta_minus")
    dp = cdf.varget(f"{species}_energy_delta_plus")
    return pd.DataFrame({
        "bin": range(len(mean)),
        "E_min": mean - dm,
        "E_mean": mean,
        "E_max": mean + dp,
    })

def bins_containing(cdf, species, energy):
    b = energy_bins(cdf, species)
    return b[(b.E_min <= energy) & (energy <= b.E_max)]

In [ ]:
# 4. Metadata: global attributes

for name, cdf in [("L2", cdf_l2), ("L3", cdf_l3)]:
    g = cdf.globalattsget()
    print(f"--- {name} ---")
    for key in ["Logical_source", "Data_version", "Generation_date", "Descriptor"]:
        print(f"{key}: {g.get(key)}")

In [ ]:
# 5a. Metadata: variables (l2)

variable_table(cdf_l2)

In [ ]:
# 5b. Metadata: variables (l3)

variable_table(cdf_l3)

In [ ]:
# 6. Energy bins: coverage by species (l2)

species_l2 = [n[:-len("_energy_mean")] for n in variable_names(cdf_l2)
              if n.endswith("_energy_mean")]

pd.DataFrame([
    {"species": s,
     "n_bins": len(b := energy_bins(cdf_l2, s)),
     "E_min": b.E_min.min(),
     "E_max": b.E_max.max()}
    for s in species_l2
])

In [ ]:
# 7. Energy bins: one species in detail (l2)

species = "h"

print(cdf_l2.varattsget(f"{species}_energy_mean").get("UNITS"))   # confirm units
energy_bins(cdf_l2, species)